# Shared CodeMirror 6 & Solveit Editor Core (`__craftCMCore`)

> Single source of truth for CM6 loading (local bundle with `esm.sh` fallback), GitHub Dark/Light syntax highlight styles, language loaders, and shared editor/DOM utilities consumed by both `craftC2` and `craftMd`.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
CONFIG = dict(
    cm_dark_pal=dict(kw='#f97583', str='#9ecbff', re='#dbedff', num='#79b8ff', fn='#b392f0', tag='#85e89d', fg='#e1e4e8', inv='#fdaeb7'),
    cm_light_pal=dict(kw='#d73a49', str='#032f62', re='#032f62', num='#005cc5', fn='#6f42c1', tag='#22863a', fg='#24292e', inv='#cb2431'),
)

## CodeMirror 6 Loader & Syntax Themes

Loads CodeMirror 6 from the optional local bundle (`/static/.craft-assets/cm6`) with automatic fallback to `esm.sh`, and builds the shared GitHub Dark/Light highlight styles used by both `craftC2` and `craftMd`.

In [ ]:
#| export_js_iife CRAFT.js cmCore
#| eval: false
  window.__craftCMCore?.disable?.();
const LOCAL_CM_BASE = '/static/.craft-assets/cm6';
  const CM_DEPS = 'deps=@codemirror/state@6.5.2,@codemirror/view@6.36.5,@codemirror/language@6.11.0,@lezer/highlight@1.2.5';
  const CM_URLS = {
    state: 'https://esm.sh/@codemirror/state@6.5.2',
    view: 'https://esm.sh/@codemirror/view@6.36.5?deps=@codemirror/state@6.5.2',
    lang: 'https://esm.sh/@codemirror/language@6.11.0?deps=@codemirror/state@6.5.2,@codemirror/view@6.36.5,@lezer/highlight@1.2.5',
    python: `https://esm.sh/@codemirror/lang-python@6.1.7?${CM_DEPS}`,
    cmds: `https://esm.sh/@codemirror/commands@6.8.1?${CM_DEPS}`,
    ac: `https://esm.sh/@codemirror/autocomplete@6.18.6?${CM_DEPS}`,
    search: 'https://esm.sh/@codemirror/search@6.5.10?deps=@codemirror/state@6.5.2,@codemirror/view@6.36.5',
    hl: 'https://esm.sh/@lezer/highlight@1.2.5',
  };

  const LANG_SPECS = {
    javascript: ['JavaScript', ['js', 'jsx'], 'lang-javascript.js', `https://esm.sh/@codemirror/lang-javascript@6?${CM_DEPS}`, m => m.javascript({jsx: true})],
    typescript: ['TypeScript', ['ts', 'tsx'], 'lang-javascript.js', `https://esm.sh/@codemirror/lang-javascript@6?${CM_DEPS}`, m => m.javascript({typescript: true, jsx: true})],
    html: ['HTML', ['htm'], 'lang-html.js', `https://esm.sh/@codemirror/lang-html@6?${CM_DEPS}`, m => m.html()],
    css: ['CSS', ['scss'], 'lang-css.js', `https://esm.sh/@codemirror/lang-css@6?${CM_DEPS}`, m => m.css()],
    json: ['JSON', ['jsonc'], 'lang-json.js', `https://esm.sh/@codemirror/lang-json@6?${CM_DEPS}`, m => m.json()],
    sql: ['SQL', ['sqlite', 'postgres', 'postgresql', 'duckdb'], 'lang-sql.js', `https://esm.sh/@codemirror/lang-sql@6?${CM_DEPS}`, m => m.sql()],
    markdown: ['Markdown', ['md'], 'lang-markdown.js', `https://esm.sh/@codemirror/lang-markdown@6.3.2?${CM_DEPS}`, m => m.markdown()],
    xml: ['XML', ['svg'], 'lang-xml.js', `https://esm.sh/@codemirror/lang-xml@6?${CM_DEPS}`, m => m.xml()],
    yaml: ['YAML', ['yml'], 'lang-yaml.js', `https://esm.sh/@codemirror/lang-yaml@6?${CM_DEPS}`, m => m.yaml()],
    rust: ['Rust', ['rs'], 'lang-rust.js', `https://esm.sh/@codemirror/lang-rust@6?${CM_DEPS}`, m => m.rust()],
    cpp: ['C++', ['c', 'cpp', 'c++', 'cxx', 'h'], 'lang-cpp.js', `https://esm.sh/@codemirror/lang-cpp@6?${CM_DEPS}`, m => m.cpp()],
    shell: ['Shell', ['bash', 'sh', 'zsh', 'shell', 'script'], 'mode-shell.js', `https://esm.sh/@codemirror/legacy-modes@6.5.1/mode/shell?${CM_DEPS}`, (m, cm) => new cm.lang.LanguageSupport(cm.lang.StreamLanguage.define(m.shell))],
  };
  const LANG_ALIASES = {
    py: 'python', py3: 'python', python3: 'python',
    ...Object.fromEntries(Object.entries(LANG_SPECS).flatMap(([k, [, aliases]]) => aliases.map(a => [a, k]))),
  };

  const isMac = navigator.userAgentData?.platform === 'macOS' || /Mac/.test(navigator.userAgent);
  const isDark = () => document.documentElement.classList.contains('dark');
  const normId = id => String(id || '').replace(/^_/, '');
  const getCellEl = id => document.getElementById('_' + normId(id));
  const normalizeText = s => String(s ?? '').replace(/\r\n?/g, '\n');

  const notifyErr = (msg, tag = 'craft') => {
    const d = el('div', {}, String(msg || 'Request failed.'));
    if (window.UIkit?.notification) window.UIkit.notification(d.innerHTML, {status: 'destructive', pos: 'bottom-right'});
    else console.error(`${tag}:`, msg);
  };

  const defineGhStyle = (cm, p, codeOnly = false) => {
    const t = cm.hl.tags;
    const rules = [
      {tag: [t.comment, t.lineComment, t.blockComment, t.docComment], color: '#6a737d'},
      {tag: [t.keyword, t.controlKeyword, t.operatorKeyword, t.definitionKeyword, t.moduleKeyword, t.operator, t.compareOperator, t.logicOperator, t.arithmeticOperator, t.bitwiseOperator, t.updateOperator], color: p.kw},
      {tag: [t.string, t.special(t.string), t.docString], color: p.str},
      {tag: [t.regexp], color: p.re},
      {tag: [t.self, t.escape, t.number, t.integer, t.float, t.bool, t.null, t.atom, t.constant(t.variableName), t.propertyName, t.attributeName, t.standard(t.variableName), t.special(t.variableName), ...(codeOnly ? [] : [t.link, t.url])], color: p.num},
      {tag: [t.function(t.variableName), t.function(t.propertyName), t.definition(t.function(t.variableName)), t.className, t.definition(t.className), t.typeName, t.meta, t.annotation], color: p.fn},
      {tag: [t.tagName], color: p.tag},
      {tag: [t.invalid], color: p.inv, fontStyle: 'italic'},
    ];
    if (!codeOnly) {
      rules.push(
        {tag: [t.heading], color: p.num, fontWeight: 'bold'},
        {tag: [t.strong], fontWeight: 'bold', color: p.fg},
        {tag: [t.emphasis], fontStyle: 'italic', color: p.fg},
        {tag: [t.variableName, t.definition(t.variableName), t.punctuation, t.bracket, t.paren, t.squareBracket, t.brace], color: p.fg},
      );
    }
    return cm.lang.HighlightStyle.define(rules);
  };

  const DARK_PAL = window.__craftCfg?.cm_dark_pal || {kw: '#f97583', str: '#9ecbff', re: '#dbedff', num: '#79b8ff', fn: '#b392f0', tag: '#85e89d', fg: '#e1e4e8', inv: '#fdaeb7'};
  const LIGHT_PAL = window.__craftCfg?.cm_light_pal || {kw: '#d73a49', str: '#032f62', re: '#032f62', num: '#005cc5', fn: '#6f42c1', tag: '#22863a', fg: '#24292e', inv: '#cb2431'};

  const buildStyles = cm => {
    if (cm.ghDarkStyle && cm.ghLightStyle && cm.mdCodeDarkStyle && cm.mdCodeLightStyle) return cm;
    cm.ghDarkStyle = defineGhStyle(cm, DARK_PAL, false);
    cm.ghLightStyle = defineGhStyle(cm, LIGHT_PAL, false);
    cm.mdCodeDarkStyle = defineGhStyle(cm, DARK_PAL, true);
    cm.mdCodeLightStyle = defineGhStyle(cm, LIGHT_PAL, true);
    cm.darkTheme = cm.view.EditorView.theme({'&': {lineHeight: 'var(--craft-code-lh, 1.2)'}, '.cm-scroller': {lineHeight: 'var(--craft-code-lh, 1.2)'}}, {dark: true});
    cm.lightTheme = cm.view.EditorView.theme({'&': {lineHeight: 'var(--craft-code-lh, 1.2)'}, '.cm-scroller': {lineHeight: 'var(--craft-code-lh, 1.2)'}}, {dark: false});
    cm.theme = cm.darkTheme;
    return cm;
  };

  const loadCoreCM = () => window.__craftCM6LocalPromise ??= (async () => {
    try {
      const mod = await import(`${LOCAL_CM_BASE}/index.js`);
      return buildStyles({...mod, _source: 'local'});
    } catch (localErr) {
      console.warn('craftC2: local CM6 bundle unavailable, falling back to esm.sh', localErr);
      const entries = await Promise.all(Object.entries(CM_URLS).map(async ([k, u]) => [k, await import(u)]));
      return buildStyles({...Object.fromEntries(entries), _source: 'cdn'});
    }
  })().catch(err => {
    window.__craftCM6LocalPromise = null;
    console.error('craftC2: failed to load CodeMirror 6', err);
    throw err;
  });

## Language Loaders, Find-Box Sync & Selection Helpers

Caches lazy language extensions (`loadLang`, `codeLanguages`), mirrors Solveit's `#find-box` search into active CM6 views (`syncFindBoxToView`), restores cursor/selection hints across DOM refreshes (`applySelection`), and installs unsaved-edit guards (`installDirtyGuards`, `jumpToLastCell`).

In [ ]:
#| export_js_iife CRAFT.js cmCore
#| eval: false
const langCache = window.__craftCM6Langs ??= new Map();
  async function loadLang(cm, langKey) {
    if (langKey === 'python') return cm.python.python();
    const spec = LANG_SPECS[langKey];
    if (!spec) return [];
    const ck = `${cm._source || 'cdn'}:${langKey}`;
    if (langCache.has(ck)) return langCache.get(ck);
    const local = cm._source === 'local';
    const mod = await import(local ? `${LOCAL_CM_BASE}/${spec[2]}` : spec[3]);
    const ext = spec[4](mod, cm);
    langCache.set(ck, ext);
    return ext;
  }

  function codeLanguages(cm) {
    const LD = cm.lang.LanguageDescription;
    return [
      LD.of({name: 'Python', alias: ['py', 'py3', 'python3'], load: async () => cm.python.python()}),
      ...Object.entries(LANG_SPECS)
        .filter(([k]) => k !== 'markdown')
        .map(([k, [name, alias]]) => LD.of({name, alias, load: () => loadLang(cm, k)})),
    ];
  }

  function syncFindBoxToView(v, cm) {
    if (!cm) return;
    const box = document.getElementById('find-box');
    if (!box || box.classList.contains('hidden')) return;
    const q = document.getElementById('search')?.value || '';
    if (!q) return;
    const isRegex = Boolean(document.getElementById('use_regex')?.checked);
    const useCase = Boolean(document.getElementById('use_case')?.checked);
    const caseSensitive = typeof window.smartCased === 'function' ? window.smartCased(q, useCase) : (useCase || /\p{Uppercase}/u.test(q));
    const replRow = document.getElementById('replace-row');
    const replace = replRow && !replRow.classList.contains('hidden') ? (document.getElementById('replace')?.value || '') : '';
    try {
      const sq = new cm.search.SearchQuery({search: q, regexp: isRegex, caseSensitive, replace});
      v.dispatch({effects: cm.search.setSearchQuery.of(sq)});
      cm.search.openSearchPanel(v);
    } catch {}
  }

  function applySelection(v, selSpec, resolveHint = null) {
    const len = v.state.doc.length;
    const clamp = n => Math.max(0, Math.min(len, n));
    if (selSpec?.anchor != null) {
      const anchor = clamp(selSpec.anchor);
      v.dispatch({selection: {anchor, head: clamp(selSpec.head ?? anchor)}});
    } else if (selSpec?.caretHint && resolveHint) {
      const matched = resolveHint(v.state.doc.toString(), selSpec.caretHint);
      if (matched != null) v.dispatch({selection: {anchor: clamp(matched)}});
      else if (selSpec.coords) {
        const pos = v.posAtCoords(selSpec.coords);
        v.dispatch({selection: {anchor: pos != null ? clamp(pos) : len}});
      }
    } else if (selSpec?.coords) {
      const pos = v.posAtCoords(selSpec.coords);
      v.dispatch({selection: {anchor: pos != null ? clamp(pos) : len}});
    } else if (selSpec?.toEnd) {
      v.dispatch({selection: {anchor: len}});
    }
  }

  const _dirtyGuards = new Set();
  let _origFollowRename = null, _followRenameWrapped = false;
  const _ensureFollowRenameWrapped = () => {
    if (_followRenameWrapped || typeof window.followRename !== 'function') return;
    _origFollowRename = window.followRename;
    _followRenameWrapped = true;
    window.followRename = async function(url) {
      for (const guard of [..._dirtyGuards]) {
        if (!(await guard())) return;
      }
      return _origFollowRename.call(this, url);
    };
  };
  const _teardownFollowRename = () => {
    if (!_followRenameWrapped) return;
    window.followRename = _origFollowRename;
    _origFollowRename = null;
    _followRenameWrapped = false;
  };
  function installDirtyGuards({dirtyIds, getContent, save, failMsg, tag = 'craft'}) {
    const onBeforeUnload = e => {
      if (!dirtyIds.size) return;
      e.preventDefault();
      e.returnValue = '';
    };
    window.addEventListener('beforeunload', onBeforeUnload);
    const guard = async () => {
      for (const k of [...dirtyIds]) {
        const content = getContent(k);
        if (content != null && !(await save(k, content))) {
          notifyErr(failMsg || 'Could not save edit before reopening renamed dialog.', tag);
          return false;
        }
      }
      return true;
    };
    _dirtyGuards.add(guard);
    _ensureFollowRenameWrapped();
    return () => {
      window.removeEventListener('beforeunload', onBeforeUnload);
      _dirtyGuards.delete(guard);
      if (!_dirtyGuards.size) _teardownFollowRename();
    };
  }

  const SOLVEIT = Object.freeze({
    sel: Object.freeze({
      dialog: '#dialog-container',
      cell: '#dialog-container > .editable',
      cellWithId: '#dialog-container > .editable[id]',
      primaryCell: '#dialog-container > .editable[data-sm="primary"]',
      selectedCell: '[data-sm="primary"],[data-sm="secondary"]',
      cardIn: ':scope > .card-in',
      cardOut: ':scope > .card-out',
      cardInType: '.card-in[data-mtype]',
      codeCard: '.card-in[data-mtype="code"]',
      noteCard: '.card-in[data-mtype="note"]',
      promptCard: '.card-in[data-mtype="prompt"]',
      rawCard: '.card-in[data-mtype="raw"]',
      cardHeader: '.uk-card-header',
      msgBody: '.msg-body',
      msgContent: '.msg-content',
      findBox: '#find-box',
      kid: '#kid',
      tblRow: '.tbl-row',
      ansiRed: '.ansi-red-fg',
      collapseBtn: '[hx-post="/collapse_"]',
      skipBtn: 'button[hx-post="/toggle_skip_"]',
      skipActive: 'button[hx-post="/toggle_skip_"] use[href="#lc-eye-off"]',
      editBtn: '.card-in [hx-post="/editor_"]',
      deleteBtn: '.uk-card-header button[title^="Delete"], .uk-card-header button[hx-post*="rm_msg"], .uk-card-header button[hx-post*="del_msg"]',
      running: '#dialog-container .editable[data-wait="true"], #dialog-container .htmx-request, #dialog-container [data-running="true"]',
    }),
    api: Object.freeze({
      addAbove: '/add_above_',
      addBelow: '/add_below_',
      setMtyp: '/set_mtyp_',
      toggleSkip: '/toggle_skip_',
      upsertMsg: '/upsert_msg_',
      rmMsg: '/rm_msg_',
      collapse: '/collapse_',
      addRunq: '/add_runq_',
      setContents: '/set_contents_',
      splitAt: '/split_at_',
      continuePrompt: '/continue_prompt_',
      editor: '/editor_',
      uploadAttachment: '/upload_attachment_',
    }),
    cardInOf: mtype => `.card-in[data-mtype="${mtype}"]`,
  });

  async function jumpToLastCell(exitFn) {
    await exitFn();
    const all = [...document.querySelectorAll(SOLVEIT.sel.cell)].filter(e => !e.hidden && getComputedStyle(e).display !== 'none');
    const last = all[all.length - 1];
    if (!last) return;
    if (window.craftC2?.manages?.(last.id) && !last.querySelector(SOLVEIT.sel.cardIn)?.classList.contains('collapsed')) {
      last.scrollIntoView({block: 'nearest'});
      window.craftC2.edit(last.id, {toEnd: true});
    } else if (window.craftMd?.manages?.(last.id)) {
      last.scrollIntoView({block: 'nearest'});
      window.craftMd.edit(last.id, {toEnd: true});
    } else {
      last.querySelector(`div[hx-post^="${SOLVEIT.api.editor}"]`)?.click();
    }
  }
  const MODAL_OPEN_SEL = '.uk-modal.uk-open, dialog.craft-overlay[open], dialog[data-craft-modal][open], .craft-overlay.open, [data-craft-modal].open';
  const BLOCKED_SEL = `input,textarea,select,[contenteditable],.monaco-editor,#full_editor,${MODAL_OPEN_SEL},[role="dialog"]`;
  const kid = () => document.getElementById('kid')?.value || '';
  const typingBlocked = (extra = '') => !!document.activeElement?.closest(extra ? `${BLOCKED_SEL},${extra}` : BLOCKED_SEL);

  const htmxPost = (url, values = {}, opts = {}) => window.htmx.ajax('POST', url, {
    source: '#kid', swap: 'none', ...opts, values: {kid: kid(), ...values},
  });

  const patch = (node, attrs) => {
    if (!node || !attrs) return node;
    for (const [k, v] of Object.entries(attrs)) {
      if (k === 'cls' || k === 'className') { if (v != null && v !== false) node.className = v; }
      else if (k === 'classes' && typeof v === 'object') { for (const [c, flag] of Object.entries(v)) node.classList.toggle(c, !!flag); }
      else if (k === 'dataset') { for (const [dk, dv] of Object.entries(v)) { if (dv == null || dv === false) delete node.dataset[dk]; else node.dataset[dk] = String(dv); } }
      else if (k === 'style' && typeof v === 'object') Object.assign(node.style, v);
      else if (k.startsWith('on') && typeof v === 'function') node.addEventListener(k.slice(2).toLowerCase(), v);
      else if (k in node && !k.includes('-') && k !== 'role' && k !== 'popover' && k !== 'popovertarget') {
        if (typeof node[k] === 'boolean') node[k] = !!v;
        else if (v != null && v !== false) node[k] = v;
      } else if (v == null || v === false) node.removeAttribute(k);
      else node.setAttribute(k, v === true ? '' : String(v));
    }
    return node;
  };

  const el = (tag, attrs = {}, ...kids) => {
    const node = patch(document.createElement(tag), attrs);
    for (const c of kids.flat()) if (c != null && c !== false) node.append(c);
    return node;
  };

  const qs = (sel, root = document) => root?.querySelector(sel) || null;
  const qsa = (sel, root = document) => root ? [...root.querySelectorAll(sel)] : [];

  const on = (target, evs, fn, opts) => {
    if (!target) return () => {};
    if (typeof evs === 'object' && !Array.isArray(evs)) {
      const offs = Object.entries(evs).map(([k, h]) => on(target, k, h, fn));
      return () => offs.forEach(f => f());
    }
    const list = Array.isArray(evs) ? evs : String(evs).trim().split(/\s+/);
    for (const ev of list) target.addEventListener(ev, fn, opts);
    return () => { for (const ev of list) target.removeEventListener(ev, fn, opts); };
  };

  const rafOnce = fn => {
    let id = 0;
    const run = () => { if (!id) id = requestAnimationFrame(() => { id = 0; fn(); }); };
    run.cancel = () => { if (id) { cancelAnimationFrame(id); id = 0; } };
    return run;
  };

  const store = {
    get: (k, def = null, s = localStorage) => { try { const v = s.getItem(k); return v === null ? def : v; } catch { return def; } },
    set: (k, v, s = localStorage) => { try { if (v === null || v === undefined) s.removeItem(k); else s.setItem(k, String(v)); } catch {} },
    del: (k, s = localStorage) => { try { s.removeItem(k); } catch {} },
    json: (k, def = null, s = localStorage) => { try { const v = s.getItem(k); return v === null ? def : JSON.parse(v); } catch { return def; } },
    setJson: (k, v, s = localStorage) => { try { s.setItem(k, JSON.stringify(v)); } catch {} },
  };

  const mountTpl = (name, {htmx = false, refs = false} = {}) => {
    const html = window.__craftTpl?.[name];
    if (!html) return null;
    const t = document.createElement('template');
    t.innerHTML = html;
    const node = t.content.firstElementChild;
    if (node && htmx) window.htmx?.process?.(node);
    if (!refs || !node) return node;
    const out = {root: node};
    if (typeof refs === 'object') {
      for (const [k, sel] of Object.entries(refs)) out[k] = node.querySelector(sel);
    } else {
      for (const n of node.querySelectorAll('[data-ref]')) out[n.dataset.ref] = n;
    }
    return out;
  };

  const offsetIn = (root, node, offset) => {
    if (!root || !node || !root.contains(node)) return null;
    try {
      const r = document.createRange();
      r.selectNodeContents(root);
      r.setEnd(node, offset);
      return r.toString().length;
    } catch { return null; }
  };

## Solveit Cell Mutations & Shared DOM Utilities

Provides HTMX cell mutation and event helpers (`saveInputContents`, `splitCellAt`, `switchCellMtype`, `runCellCmd`, `waitForHtmxSettle`, `waitForNewCell`, `onCellHtmx`, `fetchJson`), native overlay/bar helpers (`mountModal`, `mountMenu`, `mountBar`), vertical-edge detection (`isAtVerticalEdge`), and the shared coalescing DOM observer (`onDomChange`) used across CRAFT features.

In [ ]:
#| export_js_iife CRAFT.js cmCore
#| eval: false
async function saveInputContents({id, content, waitersMap, key = id, tag = 'craft'}) {
    let resolveReq;
    const statusP = new Promise(r => { resolveReq = r; });
    const waiters = waitersMap.get(key) || [];
    waiters.push(resolveReq);
    waitersMap.set(key, waiters);
    try {
      const ajaxP = htmxPost(SOLVEIT.api.setContents, {id_: '_' + id, ids: id, contents: JSON.stringify({[id]: content})})
        .then(() => statusP).catch(() => false);
      const timeoutP = new Promise(r => setTimeout(() => r(false), 8000));
      return Boolean(await Promise.race([ajaxP, timeoutP]));
    } catch (err) {
      console.error(`${tag}: save failed`, err);
      return false;
    } finally {
      const list = waitersMap.get(key);
      if (list) {
        const i = list.indexOf(resolveReq);
        if (i >= 0) list.splice(i, 1);
        if (!list.length) waitersMap.delete(key);
      }
    }
  }

  async function splitCellAt({id, view, msgType, onSuccess, tag = 'craft'}) {
    const content = view.state.doc.toString();
    const linenos = [...new Set(view.state.selection.ranges.map(r => view.state.doc.lineAt(r.from).number))];
    try {
      await htmxPost(SOLVEIT.api.splitAt, {content, linenos: JSON.stringify(linenos), id_: id, msg_type: msgType, is_input: '1'});
      onSuccess?.();
    } catch (err) {
      notifyErr('Could not split cell.', tag);
      console.error(`${tag}: split failed`, err);
    }
  }

  async function switchCellMtype({id, mtyp, onMatch, tag = 'craft'}) {
    try {
      await htmxPost(SOLVEIT.api.setMtyp, {id_: '_' + id, ids: '_' + id, mtyp});
      if (!onMatch) return;
      const matched = await waitForHtmxSettle(
        () => getCellEl(id)?.querySelector(SOLVEIT.sel.cardIn)?.dataset.mtype === mtyp,
        2500
      );
      if (matched) onMatch();
    } catch (err) {
      notifyErr('Could not change cell type.', tag);
      console.error(`${tag}: set_mtyp failed`, err);
    }
  }

  async function runCellCmd({id, cmd, after = '', tag = 'craft', failMsg = ''}) {
    const runCmd = cmd === 'continue' ? 'run' : cmd;
    try {
      await htmxPost(SOLVEIT.api.addRunq, {id_: '_' + id, ids: '_' + id, cmd: runCmd, after});
      if (cmd === 'continue') await htmxPost(SOLVEIT.api.continuePrompt, {id_: '_' + id, ids: '_' + id});
      return true;
    } catch (err) {
      console.error(`${tag}: run failed`, err);
      if (failMsg) notifyErr(failMsg, tag);
      return false;
    }
  }

  function isAtVerticalEdge(v, up, tol = 2) {
    const head = v.state.selection.main.head, doc = v.state.doc;
    const cHead = v.coordsAtPos(head), cEdge = v.coordsAtPos(up ? 0 : doc.length);
    return cHead && cEdge ? Math.abs(cHead.top - cEdge.top) <= tol : doc.lineAt(head).number === (up ? 1 : doc.lines);
  }

  const _domSubs = new Set();
  const _cellSubs = new Set();
  const _pendingCells = new Set();
  const _pendingAttrCells = new Map();
  let _cellRaf = 0;
  const _DOM_ATTRS = ['class', 'style', 'hidden', 'data-mtype', 'data-editing', 'data-sm', 'data-sm-header', 'data-c2-editing', 'data-md-editing', 'data-diff', 'data-wait', 'value'];
  let _domObs = null, _domObserving = false, _domCharData = false;
  const _inDialog = target => {
    const el = target instanceof Element ? target : target?.parentElement;
    return Boolean(el && (el.id === 'dialog-container' || el.closest(SOLVEIT.sel.dialog)));
  };
  const _flushCells = () => {
    _cellRaf = 0;
    const added = [..._pendingCells].filter(c => c.isConnected);
    const attrMap = new Map(_pendingAttrCells);
    _pendingCells.clear();
    _pendingAttrCells.clear();
    if (!added.length && !attrMap.size) return;
    for (const sub of [..._cellSubs]) {
      if (!sub.attrs) {
        if (added.length) sub.fn(added);
        continue;
      }
      const hit = new Set(added);
      for (const a of sub.attrs) {
        const s = attrMap.get(a);
        if (s) for (const c of s) if (c.isConnected) hit.add(c);
      }
      if (hit.size) sub.fn([...hit]);
    }
  };
  const _collectCells = records => {
    if (!_cellSubs.size) return;
    let dirty = false;
    for (const r of records) {
      if (r.type === 'childList') {
        for (const node of r.addedNodes) {
          if (!(node instanceof Element)) continue;
          if (node.matches(SOLVEIT.sel.cell)) { _pendingCells.add(node); dirty = true; }
          else if (node.id === 'dialog-container' || node.querySelector?.(SOLVEIT.sel.cell)) {
            for (const c of node.querySelectorAll(SOLVEIT.sel.cell)) { _pendingCells.add(c); dirty = true; }
          } else {
            const hostCell = node.closest?.(SOLVEIT.sel.cell);
            if (hostCell) { _pendingCells.add(hostCell); dirty = true; }
          }
        }
      } else if (r.type === 'attributes' && r.target instanceof Element) {
        const hostCell = r.target.closest?.(SOLVEIT.sel.cell);
        if (hostCell) {
          let s = _pendingAttrCells.get(r.attributeName);
          if (!s) _pendingAttrCells.set(r.attributeName, s = new Set());
          s.add(hostCell);
          dirty = true;
        }
      }
    }
    if (dirty && !_cellRaf) _cellRaf = requestAnimationFrame(_flushCells);
  };
  let _muting = 0;
  const _isLiveMut = r => r.type !== 'attributes' || !(r.target instanceof Element) || r.target.getAttribute(r.attributeName) !== r.oldValue;
  const _handleRecords = records => {
    const live = records.filter(_isLiveMut);
    if (!live.length) return;
    _collectCells(live);
    for (const sub of [..._domSubs]) {
      const hit = live.some(r =>
        (!sub.inDialog || _inDialog(r.target)) &&
        (r.type === 'childList' || (sub.charData && r.type === 'characterData') || (sub.attrs && sub.attrs.has(r.attributeName)))
      );
      if (!hit) continue;
      if (sub.delay === 0) sub.fn(live);
      else if (sub.delay === 'micro') { if (!sub.pend) { sub.pend = true; queueMicrotask(() => { sub.pend = false; if (_domSubs.has(sub)) _runMuted(sub.fn); }); } }
      else if (sub.delay === 'raf') { if (!sub.pend) sub.pend = requestAnimationFrame(() => { sub.pend = 0; if (_domSubs.has(sub)) _runMuted(sub.fn); }); }
      else if (typeof sub.delay === 'number') { if (!sub.pend) sub.pend = setTimeout(() => { sub.pend = 0; if (_domSubs.has(sub)) _runMuted(sub.fn); }, sub.delay); }
    }
  };
  const _runMuted = fn => {
    if (!_domObs || !_domObserving) return fn();
    if (!_muting) {
      const pre = _domObs.takeRecords();
      if (pre.length) _handleRecords(pre);
    }
    _muting++;
    try { return fn(); }
    finally {
      _muting--;
      if (!_muting && _domObs && _domObserving) _domObs.takeRecords();
    }
  };
  const _ensureDomObs = () => {
    if ((!_domSubs.size && !_cellSubs.size) || !document.body) return;
    const needCharData = [..._domSubs].some(s => s.charData);
    if (_domObserving && _domCharData === needCharData) return;
    if (!_domObs) _domObs = new MutationObserver(_handleRecords);
    if (_domObserving) _domObs.disconnect();
    _domObs.observe(document.body, {childList: true, subtree: true, attributes: true, attributeOldValue: true, characterData: needCharData, attributeFilter: _DOM_ATTRS});
    _domObserving = true;
    _domCharData = needCharData;
  };
  const _teardownDomObs = () => {
    document.removeEventListener('DOMContentLoaded', _ensureDomObs);
    if (_cellRaf) { cancelAnimationFrame(_cellRaf); _cellRaf = 0; }
    _cellSubs.clear();
    _pendingCells.clear();
    _pendingAttrCells.clear();
    for (const sub of _domSubs) {
      if (sub.delay === 'raf' && sub.pend) cancelAnimationFrame(sub.pend);
      else if (typeof sub.delay === 'number' && sub.pend) clearTimeout(sub.pend);
      sub.pend = 0;
    }
    _domSubs.clear();
    if (_domObs) { _domObs.disconnect(); _domObserving = false; _domCharData = false; }
  };
  function onDomChange(fn, {attrs = null, delay = 'raf', inDialog = false, charData = false} = {}) {
    const sub = {fn, attrs: attrs ? new Set(attrs) : null, delay, inDialog, charData, pend: 0};
    _domSubs.add(sub);
    if (document.body) _ensureDomObs();
    else document.addEventListener('DOMContentLoaded', _ensureDomObs, {once: true});
    return () => {
      _domSubs.delete(sub);
      if (sub.delay === 'raf' && sub.pend) cancelAnimationFrame(sub.pend);
      else if (typeof sub.delay === 'number' && sub.pend) clearTimeout(sub.pend);
      sub.pend = 0;
      if (!_domSubs.size && !_cellSubs.size && _domObs) { _domObs.disconnect(); _domObserving = false; _domCharData = false; }
      else if (sub.charData) _ensureDomObs();
    };
  }
  function onCells(fn, {attrs = null, initial = false} = {}) {
    const sub = {fn, attrs: attrs ? new Set(attrs) : null};
    _cellSubs.add(sub);
    if (document.body) _ensureDomObs();
    else document.addEventListener('DOMContentLoaded', _ensureDomObs, {once: true});
    if (initial) {
      const runInit = () => { const cs = allCells(); if (cs.length && _cellSubs.has(sub)) fn(cs); };
      if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', runInit, {once: true});
      else runInit();
    }
    return () => {
      _cellSubs.delete(sub);
      if (!_domSubs.size && !_cellSubs.size && _domObs) { _domObs.disconnect(); _domObserving = false; _domCharData = false; }
    };
  }

  const allCells = ({visible = false} = {}) => {
    const cells = qsa(SOLVEIT.sel.cellWithId);
    return visible ? cells.filter(c => !c.classList.contains('craft-omitted') && c.offsetParent !== null) : cells;
  };
  const selectedCells = ({includeOmitted = false} = {}) => allCells().filter(e =>
    e.matches(SOLVEIT.sel.selectedCell) && (includeOmitted || !e.classList.contains('craft-omitted'))
  );
  const primaryCell = () => qs(SOLVEIT.sel.primaryCell) || selectedCells()[0] || null;
  const cellMtype = cell => cell?.querySelector(SOLVEIT.sel.cardIn)?.dataset.mtype || 'note';
  const dlgKey = () => {
    const u = new URL(location.href);
    return u.searchParams.get('name') || u.pathname || 'default';
  };
  const sleep = ms => new Promise(r => setTimeout(r, ms));
  const notify = (msg, opts = {}) => {
    const {status = 'primary', pos = 'bottom-right', timeout = 2500} = typeof opts === 'string' ? {status: opts} : opts;
    const d = el('div', {}, String(msg || ''));
    if (window.UIkit?.notification) window.UIkit.notification(d.innerHTML, {status, pos, timeout});
    else if (status === 'danger' || status === 'destructive') console.error('craft:', msg);
    else console.info('craft:', msg);
  };
  async function deleteCellEl(cell, {tag = 'craft'} = {}) {
    if (!cell || !cell.isConnected) return false;
    const delBtn = cell.querySelector(SOLVEIT.sel.deleteBtn);
    if (delBtn) {
      delBtn.click();
      await waitForHtmxSettle(() => !cell.isConnected, 1500);
      return true;
    }
    try {
      await htmxPost(SOLVEIT.api.rmMsg, {id_: cell.id, ids: cell.id});
      return true;
    } catch (err) {
      console.error(`${tag}: delete failed`, err);
      return false;
    }
  }
  function waitForHtmxSettle(checkOrEl = null, timeoutMs = 2500) {
    const isPred = typeof checkOrEl === 'function';
    if (isPred) { const hit = checkOrEl(); if (hit) return Promise.resolve(hit); }
    return new Promise(resolve => {
      let done = false;
      const finish = val => {
        if (done) return;
        done = true;
        clearTimeout(timer);
        offSettle(); offSwap(); offDom();
        resolve(val);
      };
      const check = e => {
        if (isPred) { const hit = checkOrEl(); if (hit) finish(hit); }
        else if (!checkOrEl || e?.target === checkOrEl || checkOrEl.contains?.(e?.target)) finish(true);
      };
      const offSettle = on(document, 'htmx:afterSettle', check);
      const offSwap = on(document, 'htmx:afterSwap', check);
      const offDom = isPred ? onDomChange(() => check(), {delay: 'micro', inDialog: true}) : () => {};
      const timer = setTimeout(() => finish(isPred ? (checkOrEl() || null) : false), timeoutMs);
    });
  }

  const waitForNewCell = (beforeIds, timeoutMs = 3000) =>
    waitForHtmxSettle(() => allCells().find(c => !beforeIds.has(c.id)) || null, timeoutMs);

  function getCellInputText(cell, {rawOnly = false} = {}) {
    if (!cell) return rawOnly ? null : '';
    const id = normId(cell.id);
    const ta = cell.querySelector(`textarea[name="content"], input[name="content"], textarea#_${id}_content, textarea[data-content], textarea[name="msg"], textarea`);
    if (ta && typeof ta.value === 'string') return ta.value;
    const mtype = cellMtype(cell);
    if (mtype === 'code' || mtype === 'raw') {
      const pre = cell.querySelector(':scope > .card-in .msg-content pre code, :scope > .card-in .msg-content pre');
      if (pre) return pre.textContent || '';
    }
    if (rawOnly) return null;
    const cm = cell.querySelector('.cm-content');
    if (cm) return cm.textContent || '';
    return cell.querySelector(`${SOLVEIT.sel.cardIn} ${SOLVEIT.sel.msgContent}`)?.textContent?.trim() || '';
  }

  function headingLevel(cell) {
    const h = cell?.querySelector(`${SOLVEIT.sel.noteCard} ${SOLVEIT.sel.msgContent} :is(h1,h2,h3,h4,h5,h6)`);
    return h ? Number(h.tagName[1]) || 0 : 0;
  }

  function onCellHtmx(paths, fn, {signal = null} = {}) {
    const prefixes = Array.isArray(paths) ? paths : [paths];
    return on(document, 'htmx:configRequest', e => {
      const path = e.detail?.path || '';
      if (!prefixes.some(p => path.startsWith(p))) return;
      const params = e.detail?.parameters || {};
      const rawId = params.id_ || params.id || '';
      const id = rawId ? normId(rawId) : '';
      const ids = params.ids ? String(params.ids).split(',').map(normId).filter(Boolean) : (id ? [id] : []);
      let contents = null;
      if (params.contents) { try { contents = JSON.parse(params.contents); } catch {} }
      fn({path, params, id, ids, contents, event: e});
    }, signal ? {signal} : undefined);
  }

  async function fetchJson(url, {method = 'POST', body = null, signal = null, timeoutMs = 15000, headers = {}} = {}) {
    const ac = new AbortController();
    const onAbort = () => ac.abort();
    if (signal) {
      if (signal.aborted) ac.abort();
      else signal.addEventListener('abort', onAbort, {once: true});
    }
    const timer = timeoutMs ? setTimeout(() => ac.abort(), timeoutMs) : 0;
    try {
      const opts = {method, signal: ac.signal, headers: {...headers}};
      if (body !== null && body !== undefined) {
        if (typeof body === 'string' || body instanceof FormData || body instanceof URLSearchParams || body instanceof Blob) opts.body = body;
        else { opts.headers['Content-Type'] ||= 'application/json'; opts.body = JSON.stringify(body); }
      }
      const res = await fetch(url, opts);
      if (!res.ok) throw new Error(`HTTP ${res.status}`);
      return await res.json();
    } finally {
      clearTimeout(timer);
      if (signal) signal.removeEventListener('abort', onAbort);
    }
  }

  function mountModal(tplOrEl, {refs = true, htmx = false, closeOnBackdrop = true, closeOnEscape = true, onOpen = null, onClose = null, signal = null} = {}) {
    const mounted = typeof tplOrEl === 'string' ? mountTpl(tplOrEl, {refs, htmx}) : {root: tplOrEl};
    const root = mounted.root || mounted;
    const isDlg = root.tagName === 'DIALOG' && typeof root.showModal === 'function';
    root.setAttribute('data-craft-modal', '');
    if (!root.isConnected && document.body) document.body.append(root);
    const localAc = new AbortController();
    const onExtAbort = () => destroy();
    if (signal) {
      if (signal.aborted) localAc.abort();
      else signal.addEventListener('abort', onExtAbort, {once: true});
    }
    const sig = {signal: localAc.signal};
    const isOpen = () => isDlg ? root.open : root.classList.contains('open');
    const close = () => {
      if (!isOpen()) return;
      if (isDlg) root.close();
      else { root.classList.remove('open'); onClose?.(); }
    };
    const open = () => {
      if (!root.isConnected && document.body) document.body.append(root);
      if (isDlg) { if (!root.open) root.showModal(); }
      else root.classList.add('open');
      onOpen?.();
    };
    const destroy = () => {
      if (signal) signal.removeEventListener('abort', onExtAbort);
      localAc.abort();
      if (isDlg && root.open) root.close();
      root.remove();
    };
    if (isDlg) {
      on(root, 'close', () => onClose?.(), sig);
      on(root, 'cancel', e => {
        if (!closeOnEscape) e.preventDefault();
        else e.stopPropagation();
      }, sig);
    } else if (closeOnEscape) {
      on(document, 'keydown', e => {
        if (e.key === 'Escape' && isOpen()) { e.preventDefault(); e.stopPropagation(); close(); }
      }, {capture: true, signal: localAc.signal});
    }
    if (closeOnBackdrop) on(root, 'click', e => { if (e.target === root) close(); }, sig);
    const closeBtn = mounted.close instanceof Element ? mounted.close : null;
    if (closeBtn) on(closeBtn, 'click', close, sig);
    return {...(typeof mounted === 'object' ? mounted : {}), root, closeBtn, open, close, isOpen, destroy};
  }

  function mountMenu(tplOrEl, {refs = false, htmx = false, onOpen = null, onClose = null, signal = null} = {}) {
    const mounted = typeof tplOrEl === 'string' ? mountTpl(tplOrEl, {refs, htmx}) : {root: tplOrEl};
    const root = mounted.root || mounted;
    if (!root.hasAttribute('popover')) root.setAttribute('popover', 'auto');
    if (!root.isConnected && document.body) document.body.append(root);
    const localAc = new AbortController();
    const onExtAbort = () => destroy();
    if (signal) {
      if (signal.aborted) localAc.abort();
      else signal.addEventListener('abort', onExtAbort, {once: true});
    }
    const sig = {signal: localAc.signal};
    const isOpen = () => root.matches(':popover-open');
    const items = () => qsa('button:not(:disabled), [role="menuitem"]:not([aria-disabled="true"])', root);
    const openAt = (x, y) => {
      if (!root.isConnected && document.body) document.body.append(root);
      if (!isOpen()) root.showPopover();
      if (x != null && y != null) {
        const r = root.getBoundingClientRect();
        const pad = 8;
        root.style.left = Math.max(pad, Math.min(x, window.innerWidth - r.width - pad)) + 'px';
        root.style.top = Math.max(pad, Math.min(y, window.innerHeight - r.height - pad)) + 'px';
      }
    };
    const close = () => { if (isOpen()) root.hidePopover(); };
    const destroy = () => {
      if (signal) signal.removeEventListener('abort', onExtAbort);
      localAc.abort();
      close();
      root.remove();
    };
    on(root, 'toggle', e => {
      if (e.newState === 'open') onOpen?.();
      else onClose?.();
    }, sig);
    on(root, 'keydown', e => {
      const list = items();
      if (!list.length) return;
      const idx = list.indexOf(document.activeElement);
      if (e.key === 'ArrowDown') { e.preventDefault(); list[(idx + 1) % list.length].focus(); }
      else if (e.key === 'ArrowUp') { e.preventDefault(); list[(idx - 1 + list.length) % list.length].focus(); }
      else if (e.key === 'Home') { e.preventDefault(); list[0].focus(); }
      else if (e.key === 'End') { e.preventDefault(); list[list.length - 1].focus(); }
      else if (e.key === 'Escape') { e.stopPropagation(); close(); }
    }, sig);
    return {...(typeof mounted === 'object' ? mounted : {}), root, openAt, close, isOpen, items, destroy};
  }

  function mountBar(tplOrEl, {refs = true, htmx = false, parent = null} = {}) {
    let mounted = null;
    const ensure = () => {
      if (mounted) return mounted;
      mounted = typeof tplOrEl === 'string' ? mountTpl(tplOrEl, {refs, htmx}) : {root: tplOrEl};
      return mounted;
    };
    const show = () => {
      const m = ensure();
      const host = parent || document.body;
      if (host && !m.root.isConnected) host.append(m.root);
      m.root.hidden = false;
      return m;
    };
    const hide = () => { if (mounted?.root) mounted.root.hidden = true; };
    const destroy = () => { mounted?.root?.remove(); mounted = null; };
    const isShown = () => Boolean(mounted?.root?.isConnected && !mounted.root.hidden);
    return {ensure, show, hide, destroy, isShown, get current() { return mounted; }};
  }

  const FEATURE_KEY = 'craft-features-v1';
  const _featureRegistry = new Map();
  const _readFeatureFlags = () => {
    const raw = store.json(FEATURE_KEY, {});
    return raw && typeof raw === 'object' && !Array.isArray(raw) ? raw : {};
  };
  const features = {
    KEY: FEATURE_KEY,
    all: () => _readFeatureFlags(),
    isEnabled: id => _readFeatureFlags()[id] !== false,
    setEnabled(id, enabled) {
      const flags = _readFeatureFlags();
      if (enabled) delete flags[id];
      else flags[id] = false;
      store.setJson(FEATURE_KEY, flags);
      return enabled;
    },
    toggle: id => features.setEnabled(id, !features.isEnabled(id)),
    register(id, label = id, desc = '') {
      _featureRegistry.set(id, {id, label, desc});
      return features.isEnabled(id);
    },
    init(id, label = id, desc = '') { return features.register(id, label, desc); },
    list: () => [..._featureRegistry.values()].map(f => ({...f, enabled: features.isEnabled(f.id)})),
  };

## Shared Inline Editor Factory (`createInlineEditor`) & Export

Encapsulates the common save, blur, exit, split, type-switch, boundary-stepping, and host-attachment lifecycle shared by `craftC2` and `craftMd`, then publishes the API on `window.__craftCMCore`.

In [ ]:
#| export_js_iife CRAFT.js cmCore
#| eval: false
const _optMap = new Map();
  let _optSuppressAutoEdit = 0;
  const optimistic = {
    isOptimistic: id => Boolean(id && String(id).replace(/^_/, '').startsWith('craftopt_')),
    isPending: id => { const r = _optMap.get(normId(id)); return Boolean(r && !r.realId && !r.failed); },
    get: id => _optMap.get(normId(id)) || null,
    register(optId, promise) {
      const k = normId(optId);
      const rec = {promise, realId: null, failed: false};
      _optMap.set(k, rec);
      promise.then(realId => { if (realId) rec.realId = normId(realId); else rec.failed = true; }, () => { rec.failed = true; });
      return rec;
    },
    resolve(optId, realId) { const rec = _optMap.get(normId(optId)); if (rec) rec.realId = normId(realId); },
    fail(optId) { const rec = _optMap.get(normId(optId)); if (rec) rec.failed = true; },
    clear(optId) { _optMap.delete(normId(optId)); },
    async waitForReal(id) {
      const rec = _optMap.get(normId(id));
      if (!rec) return optimistic.isOptimistic(id) ? null : normId(id);
      if (rec.realId) return rec.realId;
      if (rec.failed) return null;
      return await rec.promise;
    },
    suppressAutoEdit() {
      _optSuppressAutoEdit++;
      let done = false;
      return () => { if (!done) { done = true; _optSuppressAutoEdit = Math.max(0, _optSuppressAutoEdit - 1); } };
    },
    isSuppressingAutoEdit: () => _optSuppressAutoEdit > 0,
  };

  function moveNodePreservingFocus(parent, node, afterEl, view) {
    const hadFocus = Boolean(view?.hasFocus || (document.activeElement && node.contains(document.activeElement)));
    const sel = hadFocus && view ? view.state.selection : null;
    const refNode = afterEl ? afterEl.nextSibling : null;
    if (typeof parent.moveBefore === 'function') {
      try { parent.moveBefore(node, refNode); }
      catch { if (afterEl) afterEl.after(node); else parent.append(node); }
    } else {
      if (afterEl) afterEl.after(node); else parent.append(node);
    }
    if (hadFocus && view && !view.hasFocus) {
      if (sel) view.dispatch({selection: sel});
      view.focus();
    }
  }


function createInlineEditor(cfg) {
    const tag = cfg.tag || 'craft';
    const keyToId = cfg.keyToId || (k => k);
    const idToInputKey = cfg.idToInputKey || (id => id);
    const normValue = cfg.normValue || (s => s);
    const saveState = cfg.saveState || ((m, k, rec) => m.set(k, rec.view.state));
    const getStateDoc = cfg.getStateDoc || (st => st.doc.toString());

    const states = new Map(), views = new Map(), pendingSaves = new Map();
    const dirtyIds = new Set(), failedSaves = new Set(), saveWaiters = new Map();
    const requests = new WeakMap();
    let activeKey = null, destroyed = false;

    const syncSaveState = key => {
      const st = failedSaves.has(key) ? 'error' : (pendingSaves.has(key) ? 'saving' : (dirtyIds.has(key) ? 'dirty' : ''));
      cfg.applySaveState(key, st);
    };

    async function resolveOptimisticKey(key) {
      const id = keyToId(key);
      if (!optimistic.isOptimistic(id)) return {key, id};
      const realId = await optimistic.waitForReal(id);
      return realId ? {key: idToInputKey(realId), id: realId} : null;
    }

    async function saveCell(key, content) {
      const resolved = await resolveOptimisticKey(key);
      if (!resolved) return false;
      const {key: curKey, id} = resolved;
      const el = getCellEl(id);
      if (!el) return true;
      const prev = cfg.getText(el, curKey);
      if (content === prev) {
        dirtyIds.delete(curKey);
        failedSaves.delete(curKey);
        syncSaveState(curKey);
        return pendingSaves.get(curKey) ? await pendingSaves.get(curKey) : true;
      }
      dirtyIds.add(curKey);
      failedSaves.delete(curKey);
      syncSaveState(curKey);
      const p = (async () => {
        let ok = false;
        try {
          ok = cfg.persistSave
            ? await cfg.persistSave(curKey, content, saveWaiters)
            : await saveInputContents({id, content, waitersMap: saveWaiters, key: curKey, tag});
        } finally {
          if (pendingSaves.get(curKey) === p) pendingSaves.delete(curKey);
        }
        if (ok) {
          failedSaves.delete(curKey);
          const curEl = getCellEl(id);
          const inp = cfg.getInput(curEl, curKey);
          if (inp && normValue(inp.value) === prev) inp.value = content;
          const curView = views.get(curKey)?.view;
          if (!curView || curView.state.doc.toString() === content) dirtyIds.delete(curKey);
        } else {
          failedSaves.add(curKey);
          notifyErr(cfg.saveFailMsg ? cfg.saveFailMsg(curKey) : 'Could not save cell edit — changes kept in editor.', tag);
        }
        syncSaveState(curKey);
        return ok;
      })();
      pendingSaves.set(curKey, p);
      syncSaveState(curKey);
      return await p;
    }

    function unmountCell(key) {
      const rec = views.get(key);
      cfg.beforeUnmount?.(key, rec);
      if (!rec) return;
      saveState(states, key, rec);
      const el = getCellEl(keyToId(key)), card = cfg.getCard(el, key);
      rec.view.destroy();
      rec.host.remove();
      views.delete(key);
      card?.classList.remove(cfg.mountedCls);
      cfg.setEditing(key, false, el, card);
      if (activeKey === key) activeKey = null;
    }

    async function onCellBlur(key) {
      const rec = views.get(key);
      cfg.beforeBlur?.(key, rec);
      if (!rec) return;
      const ref = rec.ref;
      const el = getCellEl(keyToId(ref.key)), card = cfg.getCard(el, ref.key);
      cfg.setEditing(ref.key, false, el, card);
      if (activeKey === ref.key) activeKey = null;
      const content = rec.view.state.doc.toString();
      saveState(states, ref.key, rec);
      const saved = await saveCell(ref.key, content);
      const curKey = ref.key, cur = views.get(curKey);
      if (!cur || cur.view.hasFocus || activeKey === curKey) return;
      if (!saved || dirtyIds.has(curKey) || cfg.keepMounted?.(curKey, cur)) return;
      unmountCell(curKey);
    }

    async function exitCell(key) {
      if (!key) return;
      const rec = views.get(key);
      cfg.beforeExit?.(key, rec);
      const el = getCellEl(keyToId(key));
      if (rec) {
        rec.view.contentDOM.blur();
        if (el) window.selectMsg?.(window.$(el), {scroll: false});
        await onCellBlur(key);
      }
      if (document.activeElement && document.activeElement !== document.body) document.activeElement.blur();
    }

    async function saveAndRun(key, cmd) {
      const rec = views.get(key);
      if (!rec) return;
      const ref = rec.ref;
      cfg.beforeRun?.(ref.key, rec);
      const content = rec.view.state.doc.toString();
      saveState(states, ref.key, rec);
      const saved = await saveCell(ref.key, content);
      if (!saved) return;
      const curKey = ref.key, id = keyToId(curKey), el = getCellEl(id);
      if (!el) return;
      const next = cmd === 'shift-run' ? window.adjacentMsg?.(window.$(el), false) : null;
      const after = cmd === 'alt-run' ? (window.sectionTail?.(el) || '') : '';
      cfg.setEditing(curKey, false, el, cfg.getCard(el, curKey));
      if (activeKey === curKey) activeKey = null;
      rec.view.contentDOM.blur();
      if (cmd === 'shift-run' && next?.length) window.selectMsg?.(next);
      else window.selectMsg?.(window.$(el), {scroll: false});
      const cur = views.get(curKey);
      if (cur && !cur.view.hasFocus && activeKey !== curKey && !dirtyIds.has(curKey) && !cfg.keepMounted?.(curKey, cur)) unmountCell(curKey);
      if (await runCellCmd({id, cmd, after, tag, failMsg: cfg.runFailMsg || ''})) schedule();
    }

    async function splitCell(key, v, msgType) {
      const resolved = await resolveOptimisticKey(key);
      if (!resolved) return;
      const {key: curKey, id} = resolved;
      cfg.beforeSplit?.(curKey, v);
      await splitCellAt({
        id, view: v, msgType, tag,
        onSuccess: () => { dirtyIds.delete(curKey); failedSaves.delete(curKey); syncSaveState(curKey); unmountCell(curKey); },
      });
    }

    async function switchCellType(key, v, mtyp, {clearState = false, onMatch} = {}) {
      const resolved = await resolveOptimisticKey(key);
      if (!resolved) return;
      const {key: curKey, id} = resolved;
      const sel = v.state.selection.main;
      const selSpec = {anchor: sel.anchor, head: sel.head};
      cfg.beforeSwitch?.(curKey, v);
      const saved = await saveCell(curKey, v.state.doc.toString());
      if (!saved) return;
      unmountCell(curKey);
      if (clearState) states.delete(curKey);
      const el = getCellEl(id);
      if (el) window.selectMsg?.(window.$(el), {scroll: false});
      await switchCellMtype({id, mtyp, tag, onMatch: onMatch ? () => onMatch(id, selSpec) : null});
    }

    function stepBoundary(key, v, up, {tol = 2, canStep, beforeStep} = {}) {
      if (v.state.selection.ranges.length !== 1 || !v.state.selection.main.empty) return false;
      if (canStep && !canStep(v)) return false;
      if (!isAtVerticalEdge(v, up, tol)) return false;
      const id = keyToId(key), el = getCellEl(id);
      const next = el && window.adjacentMsg?.(window.$(el), up);
      const nextEl = next?.[0];
      if (!nextEl || nextEl === el) return false;
      beforeStep?.(key, v);
      const rec = views.get(key), ref = rec?.ref;
      if (rec) saveState(states, key, {...rec, view: v});
      cfg.setEditing(key, false, el, cfg.getCard(el, key));
      if (activeKey === key) activeKey = null;
      v.contentDOM.blur();
      saveCell(key, v.state.doc.toString()).then(ok => {
        const curKey = ref?.key || key, cur = views.get(curKey);
        if (ok && cur && !cur.view.hasFocus && activeKey !== curKey && !dirtyIds.has(curKey) && !cfg.keepMounted?.(curKey, cur)) unmountCell(curKey);
      });
      const nextId = normId(nextEl.id);
      const edgeSel = up ? {toEnd: true} : {anchor: 0, head: 0};
      const codeCard = nextEl.querySelector(SOLVEIT.sel.codeCard);
      if (window.craftC2?.manages?.(nextId) && codeCard && !codeCard.classList.contains('collapsed')) {
        nextEl.scrollIntoView({block: 'nearest'});
        window.craftC2.edit(nextId, edgeSel);
      } else if (window.craftMd?.manages?.(nextId)) {
        nextEl.scrollIntoView({block: 'nearest'});
        window.craftMd.edit(nextId, edgeSel);
      } else {
        window.selectMsg?.(next);
      }
      return true;
    }

    function openView(key, {el, card, text, cm, selSpec = null, resolveHint = null, canReuseRec, prepareState, hostClass, extraRec, themeExts, onExisting, onMounted}) {
      const body = card?.querySelector(SOLVEIT.sel.msgBody), contentEl = body?.querySelector(SOLVEIT.sel.msgContent);
      if (!body || !contentEl) return false;
      window.selectMsg?.(window.$(el), {scroll: false});
      let rec = views.get(key);
      if (rec && (!canReuseRec || canReuseRec(rec))) {
        if (!body.contains(rec.host)) { contentEl.after(rec.host); card.classList.add(cfg.mountedCls); }
        if (rec.view.state.doc.toString() !== text && !pendingSaves.has(key) && !dirtyIds.has(key)) {
          rec.view.dispatch({changes: {from: 0, to: rec.view.state.doc.length, insert: text}});
        }
        if (selSpec) applySelection(rec.view, selSpec, resolveHint);
        activeKey = key;
        cfg.setEditing(key, true, el, card);
        syncFindBoxToView(rec.view, cm);
        rec.view.focus();
        onExisting?.(rec);
        return true;
      }
      if (rec) unmountCell(key);
      const ref = {key, id: keyToId(key)};
      const {edState, activeComps, reusePrev} = prepareState(ref);
      const host = document.createElement('div');
      host.className = hostClass;
      contentEl.after(host);
      card.classList.add(cfg.mountedCls);
      cfg.setEditing(key, true, el, card);
      const v = new cm.view.EditorView({state: edState, parent: host});
      rec = {view: v, host, ref, ...activeComps, ...extraRec};
      attachHost(rec, host);
      views.set(key, rec);
      activeKey = key;
      cfg.setEditing(key, true, el, card);
      if (reusePrev && themeExts && activeComps.themeComp) v.dispatch({effects: activeComps.themeComp.reconfigure(themeExts)});
      onMounted?.(rec);
      applySelection(v, selSpec || (reusePrev ? null : {toEnd: true}), resolveHint);
      syncFindBoxToView(v, cm);
      v.focus();
      requestAnimationFrame(() => {
        const curKey = ref.key;
        if (!destroyed && activeKey === curKey && views.get(curKey)?.view === v && !v.hasFocus && document.activeElement === document.body) v.focus();
      });
      return true;
    }

    function attachHost(rec, host) {
      host.addEventListener('keydown', e => { e.stopPropagation(); });
      host.addEventListener('focusin', () => {
        const key = rec.ref.key;
        activeKey = key;
        const curEl = getCellEl(keyToId(key));
        cfg.setEditing(key, true, curEl, cfg.getCard(curEl, key));
      });
      host.addEventListener('focusout', () => {
        const check = () => {
          if (destroyed || rec._reparenting) return;
          if (!host.isConnected) { sync(); return; }
          const ae = document.activeElement;
          if (host.contains(ae) || ae?.closest?.('.cm-tooltip')) return;
          onCellBlur(rec.ref.key);
        };
        queueMicrotask(() => {
          if (destroyed || rec._reparenting) return;
          if (!host.isConnected) { sync(); return; }
          requestAnimationFrame(check);
        });
      });
    }

    function rekey(oldKey, newKey, {el, card} = {}) {
      const rec = views.get(oldKey);
      if (!rec) return false;
      const oldId = keyToId(oldKey), newId = keyToId(newKey);
      const oldEl = getCellEl(oldId), oldCard = cfg.getCard(oldEl, oldKey);
      views.delete(oldKey);
      views.set(newKey, rec);
      rec.ref.key = newKey;
      rec.ref.id = newId;
      if (states.has(oldKey)) { states.set(newKey, states.get(oldKey)); states.delete(oldKey); }
      if (dirtyIds.has(oldKey)) { dirtyIds.delete(oldKey); dirtyIds.add(newKey); }
      if (failedSaves.has(oldKey)) { failedSaves.delete(oldKey); failedSaves.add(newKey); }
      if (pendingSaves.has(oldKey)) { pendingSaves.set(newKey, pendingSaves.get(oldKey)); pendingSaves.delete(oldKey); }
      if (saveWaiters.has(oldKey)) { saveWaiters.set(newKey, saveWaiters.get(oldKey)); saveWaiters.delete(oldKey); }
      if (activeKey === oldKey) activeKey = newKey;
      cfg.onRekey?.(oldKey, newKey, rec);
      const realEl = el || getCellEl(newId), realCard = card || cfg.getCard(realEl, newKey);
      const body = realCard?.querySelector(SOLVEIT.sel.msgBody), contentEl = body?.querySelector(SOLVEIT.sel.msgContent);
      if (body && contentEl) {
        rec._reparenting = true;
        try { moveNodePreservingFocus(body, rec.host, contentEl, rec.view); }
        finally { queueMicrotask(() => { rec._reparenting = false; }); }
        realCard.classList.add(cfg.mountedCls);
        oldCard?.classList.remove(cfg.mountedCls);
        cfg.setEditing(oldKey, false, oldEl, oldCard);
        if (activeKey === newKey || rec.view.hasFocus) cfg.setEditing(newKey, true, realEl, realCard);
      }
      syncSaveState(newKey);
      return true;
    }

    const sync = () => {
      cfg.onSync?.();
      for (const [key, rec] of [...views]) {
        const el = getCellEl(keyToId(key)), card = cfg.getCard(el, key);
        if (!el || !card || !cfg.isViewValid(key, rec, el, card)) {
          rec.view.destroy();
          views.delete(key);
          dirtyIds.delete(key);
          failedSaves.delete(key);
          syncSaveState(key);
          if (activeKey === key) activeKey = null;
          continue;
        }
        const body = card.querySelector(SOLVEIT.sel.msgBody), contentEl = body?.querySelector(SOLVEIT.sel.msgContent);
        if (body && contentEl) {
          if (!body.contains(rec.host)) contentEl.after(rec.host);
          card.classList.add(cfg.mountedCls);
          if (activeKey === key) {
            cfg.setEditing(key, true, el, card);
            if (!rec.view.hasFocus || document.activeElement !== rec.view.contentDOM) rec.view.focus();
          }
        }
        cfg.onViewSynced?.(key, rec);
      }
      for (const key of [...states.keys()]) {
        if (!views.has(key) && !dirtyIds.has(key) && !pendingSaves.has(key) && !getCellEl(keyToId(key))) states.delete(key);
      }
    };
    const schedule = rafOnce(sync);

    const before = ({detail: {ctx} = {}}) => {
      const request = ctx?.request, body = request?.body;
      if (!body?.get || request.method !== 'POST') return;
      const pathname = new URL(request.action, location.href).pathname;
      if (pathname === SOLVEIT.api.setContents) {
        const id = normId(body.get('ids'));
        const key = id ? idToInputKey(id) : '';
        const waiters = key ? saveWaiters.get(key) : null;
        if (waiters?.length) {
          const resolve = waiters.shift();
          if (!waiters.length) saveWaiters.delete(key);
          requests.set(ctx, {kind: 'save', resolve});
        }
        return;
      }
      cfg.onBeforeRequest?.(ctx, pathname, body);
    };

    const after = ({detail: {ctx} = {}}) => {
      const rec = ctx && requests.get(ctx);
      if (rec?.kind === 'save') {
        const ok = Boolean(ctx.response && ctx.response.status >= 200 && ctx.response.status < 300);
        rec.resolve?.(ok);
        return;
      }
      cfg.onAfterRequest?.(ctx);
    };

    const onKeyDown = e => {
      if (!cfg.openPrimaryOnEnter) return;
      if (e.key !== 'Enter' || e.shiftKey || e.ctrlKey || e.metaKey || e.altKey || e.isComposing) return;
      if (typingBlocked(cfg.typingBlockedExtra || '')) return;
      const el = document.querySelector(SOLVEIT.sel.primaryCell);
      if (!el || !cfg.openPrimaryOnEnter(el, false)) return;
      e.preventDefault();
      e.stopPropagation();
      e.stopImmediatePropagation();
      cfg.openPrimaryOnEnter(el, true);
    };

    const onClick = e => {
      if (cfg.onEditClick) {
        const edit = e.target.closest?.(cfg.editClickSel || SOLVEIT.sel.editBtn);
        if (edit) {
          if (cfg.onEditClick(edit, e)) return;
        }
      }
      cfg.onExtraClick?.(e);
    };

    const onThemeChange = () => {
      if (!cfg.getThemeExts) return;
      for (const rec of views.values()) {
        const exts = cfg.getThemeExts(rec);
        if (exts && rec.themeComp) rec.view.dispatch({effects: rec.themeComp.reconfigure(exts)});
      }
    };

    const DEFAULT_POINTER_IGNORE = 'a, button, summary, input, select, textarea, label, [role="button"], .craft-cm-host, .copy-wrap > button';
    let pendingPointer = null;
    const onPointerDown = cfg.pointerEdit ? e => {
      const pe = cfg.pointerEdit;
      if (e.button !== 0) return;
      if (pe.ignoreMods && (e.metaKey || e.ctrlKey || e.altKey)) return;
      if (e.target.closest?.(pe.ignoreSel || DEFAULT_POINTER_IGNORE)) return;
      const hit = pe.resolveTarget(e);
      if (!hit) return;
      pendingPointer = {...hit, x: e.clientX, y: e.clientY};
    } : null;
    const onPointerUp = cfg.pointerEdit ? e => {
      if (!pendingPointer) return;
      const p = pendingPointer;
      pendingPointer = null;
      const sel = window.getSelection();
      if (p.dragSelectGuard) {
        const moved = Math.hypot(e.clientX - p.x, e.clientY - p.y);
        if (moved > 5 && sel && !sel.isCollapsed) return;
        cfg.pointerEdit.onOpen(p, {caretHint: p.caretHint, coords: {x: e.clientX, y: e.clientY}}, e);
        return;
      }
      const anchor = offsetIn(p.codeEl, sel?.anchorNode, sel?.anchorOffset);
      const head = offsetIn(p.codeEl, sel?.focusNode, sel?.focusOffset);
      if (anchor != null) sel?.removeAllRanges();
      cfg.pointerEdit.onOpen(p, anchor != null ? {anchor, head: head ?? anchor} : {coords: {x: e.clientX, y: e.clientY}}, e);
    } : null;

    const sep = window.htmx?.config.metaCharacter || ':';
    const configEvent = ['htmx', 'config', 'request'].join(sep);
    const afterEvent = ['htmx', 'after', 'request'].join(sep);
    const unsubDom = onDomChange(schedule, {attrs: cfg.domAttrs || null, delay: 'raf', inDialog: Boolean(cfg.inDialog)});
    const removeDirtyGuards = installDirtyGuards({
      dirtyIds,
      getContent: key => (views.get(key) ? views.get(key).view.state.doc.toString() : (states.get(key) ? getStateDoc(states.get(key)) : undefined)),
      save: saveCell,
      failMsg: cfg.failRenameMsg || 'Could not save edit before reopening renamed dialog.',
      tag,
    });
    const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
    on(document, {[configEvent]: before, [afterEvent]: after, 'uk-theme-switcher:change': onThemeChange}, sig);
    on(document, {click: onClick, keydown: onKeyDown}, cap);
    if (onPointerDown && onPointerUp) on(document, {mousedown: onPointerDown, mouseup: onPointerUp}, sig);

    return {
      states, views, pendingSaves, dirtyIds, failedSaves, saveWaiters,
      get activeKey() { return activeKey; },
      set activeKey(v) { activeKey = v; },
      get destroyed() { return destroyed; },
      syncSaveState, saveCell, unmountCell, onCellBlur, exitCell, saveAndRun, splitCell,
      switchCellType, stepBoundary, openView, attachHost, rekey, sync, schedule,
      disable() {
        destroyed = true;
        unsubDom();
        schedule.cancel();
        removeDirtyGuards();
        pendingPointer = null;
        ac.abort();
        for (const key of [...views.keys()]) unmountCell(key);
        views.clear(); states.clear(); pendingSaves.clear(); dirtyIds.clear(); failedSaves.clear(); saveWaiters.clear();
      },
    };
  }

## Export `window.__craftCMCore`

In [ ]:
#| export_js_iife CRAFT.js cmCore
#| eval: false
  /**
   * @typedef {Object} CraftSolveitContract
   * @property {Readonly<Record<string, string>>} sel
   * @property {Readonly<Record<string, string>>} api
   * @property {(mtype: string) => string} cardInOf
   */
  /**
   * @typedef {Object} CraftFeatureInfo
   * @property {string} id
   * @property {string} label
   * @property {string} desc
   * @property {boolean} enabled
   */
  /**
   * @typedef {Object} CraftFeaturesApi
   * @property {string} KEY
   * @property {() => Record<string, boolean>} all
   * @property {(id: string) => boolean} isEnabled
   * @property {(id: string, enabled: boolean) => boolean} setEnabled
   * @property {(id: string) => boolean} toggle
   * @property {(id: string, label?: string, desc?: string) => boolean} register
   * @property {(id: string, label?: string, desc?: string) => boolean} init
   * @property {() => CraftFeatureInfo[]} list
   */
  /**
   * @typedef {Object} CraftPointerEditConfig
   * @property {string} [ignoreSel]
   * @property {boolean} [ignoreMods]
   * @property {(e: MouseEvent) => ({id: string, codeEl?: Element|null, caretHint?: any, dragSelectGuard?: boolean} | null)} resolveTarget
   * @property {(target: any, selSpec: any, e: MouseEvent) => void} onOpen
   */
  /**
   * @typedef {Object} CraftCMCore
   * @property {string} LOCAL_CM_BASE
   * @property {string} CM_DEPS
   * @property {Record<string, string>} CM_URLS
   * @property {Record<string, any>} LANG_SPECS
   * @property {Record<string, string>} LANG_ALIASES
   * @property {Map<string, any>} langCache
   * @property {CraftSolveitContract} SOLVEIT
   * @property {CraftFeaturesApi} features
   * @property {boolean} isMac
   * @property {() => boolean} isDark
   * @property {(id: any) => string} normId
   * @property {(id: any) => HTMLElement|null} getCellEl
   * @property {(s: any) => string} normalizeText
   * @property {(msg: string, tag?: string) => void} notifyErr
   * @property {(cm: any) => any} buildStyles
   * @property {() => Promise<any>} loadCoreCM
   * @property {(cm: any, langKey: string) => Promise<any>} loadLang
   * @property {(code: string) => string[]} codeLanguages
   * @property {(view: any, cm: any) => void} syncFindBoxToView
   * @property {(view: any, selSpec: any, resolveHint?: any) => void} applySelection
   * @property {(opts: any) => () => void} installDirtyGuards
   * @property {(exitFn: () => Promise<any>) => Promise<void>} jumpToLastCell
   * @property {string} MODAL_OPEN_SEL
   * @property {() => string} kid
   * @property {(extra?: string) => boolean} typingBlocked
   * @property {(url: string, values?: Record<string, any>, opts?: Record<string, any>) => Promise<any>} htmxPost
   * @property {(node: any, attrs: Record<string, any>) => any} patch
   * @property {(tag: string, attrs?: Record<string, any>, ...kids: any[]) => HTMLElement} el
   * @property {(sel: string, root?: ParentNode|null) => Element|null} qs
   * @property {(sel: string, root?: ParentNode|null) => Element[]} qsa
   * @property {(target: any, evs: any, fn: any, opts?: any) => () => void} on
   * @property {(fn: () => void) => ((() => void) & {cancel: () => void})} rafOnce
   * @property {{get: Function, set: Function, del: Function, json: Function, setJson: Function}} store
   * @property {(name: string, opts?: {htmx?: boolean, refs?: boolean|Record<string, string>}) => any} mountTpl
   * @property {(root: Node|null, node: Node|null|undefined, offset: number|undefined) => number|null} offsetIn
   * @property {(opts: any) => Promise<boolean>} saveInputContents
   * @property {(opts: any) => Promise<void>} splitCellAt
   * @property {(opts: any) => Promise<void>} switchCellMtype
   * @property {(opts: any) => Promise<boolean>} runCellCmd
   * @property {(v: any, up: boolean, tol?: number) => boolean} isAtVerticalEdge
   * @property {(fn: (records?: MutationRecord[]) => void, opts?: {attrs?: string[]|null, delay?: number|'raf'|'micro', inDialog?: boolean, charData?: boolean}) => () => void} onDomChange
   * @property {(fn: (cells: HTMLElement[]) => void, opts?: {attrs?: string[]|null, initial?: boolean}) => () => void} onCells
   * @property {(cfg: any) => any} createInlineEditor
   * @property {Record<string, any>} optimistic
   * @property {(opts?: {visible?: boolean}) => HTMLElement[]} allCells
   * @property {(opts?: {includeOmitted?: boolean}) => HTMLElement[]} selectedCells
   * @property {() => HTMLElement|null} primaryCell
   * @property {(cell: Element|null) => string} cellMtype
   * @property {() => string} dlgKey
   * @property {(ms: number) => Promise<void>} sleep
   * @property {(msg: string, opts?: string|Record<string, any>) => void} notify
   * @property {(cell: Element|null, opts?: {tag?: string}) => Promise<boolean>} deleteCellEl
   * @property {(tplOrEl: string|Element, opts?: Record<string, any>) => any} mountModal
   * @property {(tplOrEl: string|Element, opts?: Record<string, any>) => any} mountMenu
   * @property {(tplOrEl: string|Element, opts?: Record<string, any>) => any} mountBar
   * @property {(checkOrEl?: any, timeoutMs?: number) => Promise<any>} waitForHtmxSettle
   * @property {(beforeIds: Set<string>, timeoutMs?: number) => Promise<HTMLElement|null>} waitForNewCell
   * @property {(cell: Element|null, opts?: {rawOnly?: boolean}) => string|null} getCellInputText
   * @property {(cell: Element|null) => number} headingLevel
   * @property {(paths: string|string[], fn: (ev: any) => void, opts?: {signal?: AbortSignal|null}) => () => void} onCellHtmx
   * @property {(url: string, opts?: Record<string, any>) => Promise<any>} fetchJson
   * @property {() => void} disable
   */
  /** @type {CraftCMCore} */
  window.__craftCMCore = {
    LOCAL_CM_BASE, CM_DEPS, CM_URLS, LANG_SPECS, LANG_ALIASES, langCache,
    SOLVEIT, features,
    isMac, isDark, normId, getCellEl, normalizeText, notifyErr,
    buildStyles, loadCoreCM, loadLang, codeLanguages, syncFindBoxToView, applySelection,
    installDirtyGuards, jumpToLastCell,
    MODAL_OPEN_SEL, kid, typingBlocked, htmxPost, patch, el, qs, qsa, on, rafOnce, store, mountTpl, offsetIn,
    saveInputContents, splitCellAt, switchCellMtype, runCellCmd, isAtVerticalEdge,
    onDomChange, onCells, createInlineEditor, optimistic, allCells, selectedCells, primaryCell, cellMtype, dlgKey, sleep, notify, deleteCellEl, mountModal, mountMenu, mountBar, waitForHtmxSettle, waitForNewCell, getCellInputText, headingLevel, onCellHtmx, fetchJson,
    disable() { _teardownDomObs(); _dirtyGuards.clear(); _teardownFollowRename(); },
  };

Verify `04_cm_core.ipynb` exports `window.__craftCMCore` into `CRAFT.js`.

## Shared CodeMirror 6 Base CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Shared CodeMirror 6 editor base (.craft-cm-host on C2 & Markdown/Raw) ── */
.craft-cm-host { position: relative; min-height: 18px; }
.craft-cm-host .cm-editor, .craft-cm-host .cm-editor.cm-focused { background: transparent !important; outline: none !important; }
.craft-cm-host .cm-scroller { font-family: inherit !important; font-size: inherit !important; overflow: visible !important; }
.craft-cm-host :is(.cm-cursor, .cm-dropCursor) { border-left: 1.5px solid var(--craft-fg) !important; }
.craft-cm-host :is(.cm-selectionBackground, .cm-editor.cm-focused .cm-selectionBackground),
.craft-cm-host ::selection { background: #7aa2f740 !important; }
.craft-cm-host .cm-selectionMatch { background: #7aa2f722 !important; }
.craft-cm-host .cm-placeholder { color: var(--craft-placeholder); font-style: italic; pointer-events: none; }
.craft-cm-host .cm-panels { background: var(--craft-recessed) !important; color: var(--craft-fg); border: 0; border-top: 1px solid var(--craft-border); }
.craft-cm-host .cm-search { padding: 4px 6px; font-family: var(--craft-prose-family); font-size: 11px; display: flex; flex-wrap: wrap; align-items: center; gap: 4px; }
.craft-cm-host .cm-search :is(input, button) { background: var(--craft-bg); color: var(--craft-fg); border: 1px solid var(--craft-border); border-radius: 4px; padding: 2px 6px; font-size: 11px; }
.craft-cm-host .cm-search button:hover { background: var(--craft-hover); }
.craft-cm-host .cm-search button[name="close"] { margin-left: auto; border: 0; background: transparent; cursor: pointer; }
.craft-cm-host .cm-searchMatch { background: #e0af6833; outline: 1px solid #e0af6866; }
.craft-cm-host .cm-searchMatch-selected { background: #7aa2f744; outline: 1px solid var(--craft-accent); }

/* Shared save-status dot & error bar */
.editable[data-c2-save="error"] > .card-in[data-mtype="code"],
.editable > .msg-card[data-md-save="error"] { box-shadow: inset 2px 0 0 var(--craft-amber) !important; }
.editable[data-c2-save] > .card-in[data-mtype="code"]::after,
.editable > .msg-card[data-md-save]::after { content: ""; position: absolute; right: 4px; bottom: 3px; width: 5px; height: 5px; border-radius: 50%; pointer-events: none; z-index: 6; }
.editable[data-c2-save="dirty"] > .card-in[data-mtype="code"]::after,
.editable > .msg-card[data-md-save="dirty"]::after { background: var(--craft-accent-muted); }
.editable[data-c2-save="saving"] > .card-in[data-mtype="code"]::after,
.editable > .msg-card[data-md-save="saving"]::after { background: var(--craft-cyan); animation: craft-c2-pulse 1s ease-in-out infinite; }
.editable[data-c2-save="error"] > .card-in[data-mtype="code"]::after,
.editable > .msg-card[data-md-save="error"]::after { background: var(--craft-amber); }
@keyframes craft-c2-pulse { 0%, 100% { opacity: .35; } 50% { opacity: 1; } }

/* Soft-wrap long lines (CM6 reads white-space from the DOM, so this is a real wrap, not a clip). */
html.craft-mobile .craft-cm-host .cm-content { white-space: pre-wrap !important; overflow-wrap: anywhere; word-break: normal; flex-shrink: 1 !important; min-width: 0; }
html.craft-mobile .craft-cm-host .cm-search :is(input, button) { font-size: 14px; padding: 6px 8px; }

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets
from fastcore.test import test_eq

assets = extract_nb_assets(REPO_ROOT/'nbs'/'craft'/'04_cm_core.ipynb')
test_eq(sorted(assets), ['CRAFT.css', 'CRAFT.js'])
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.__craftCMCore', 'const SOLVEIT', 'function onCells', 'const features', '@typedef {Object} CraftCMCore'): assert sym in js, sym